# Static Semantics-Aware Security Steering: Large-Scale Benchmark (14,000 Executions)

This notebook runs the 3,500-task, 4-condition (14,000 forward passes) benchmark on `mistralai/Mistral-7B-Instruct-v0.3`.

**Experimental Conditions:**
1. `unsteered_baseline` (Unhooked)
2. `hooked_alpha_0` (Hooked parity check)
3. `learned_vector_layer16` (Learned static vector, Layer 16, $\alpha=1.0$)
4. `random_vector_layer16` (Random control vector, Layer 16, $\alpha=1.0$)

All generations are checkpointed to `outputs/large_scale_evaluation_per_sample.csv` with on-the-fly patched security and functional evaluations.

In [ ]:
# Step 1: Verify GPU Environment (Requires T4 or higher)
!nvidia-smi


In [ ]:
# Step 2: Install Required Dependencies
!pip install -q torch transformers accelerate bitsandbytes pytest pyyaml


In [ ]:
# Step 3: Run Quick Preflight Validation (20 Tasks across all 4 conditions = 80 generations)
!python scripts/run_large_scale_evaluation.py \
    --tasks-file data/processed/expanded_evaluation_3k.json \
    --output-dir outputs \
    --preflight 20


In [ ]:
# Step 4: Run Full Benchmark (Batch 1 of 875 tasks, or omit --batch-size for full 3,500 tasks)
!python scripts/run_large_scale_evaluation.py \
    --tasks-file data/processed/expanded_evaluation_3k.json \
    --output-dir outputs \
    --batch-size 875


In [ ]:
# Step 5: Checkpoint Status & Summary Statistics
import csv
import os

csv_path = 'outputs/large_scale_evaluation_per_sample.csv'
if os.path.exists(csv_path):
    with open(csv_path, 'r', encoding='utf-8') as f:
        rows = list(csv.DictReader(f))
    print(f'Total completed evaluations: {len(rows)}')
    from collections import Counter
    print('Conditions breakdown:', Counter(r['condition'] for r in rows))
else:
    print('Checkpoint file not found yet.')
